# weight-decay-l2-add composite — cx18: L2-add WD vs decoupled WD — different effective updates at nonzero g

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `weight-decay-l2-add`, `weight-decay-decoupled`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "weight-decay-l2-add"
DD_ATOM_IDS = ["weight-decay-l2-add", "weight-decay-decoupled"]
DD_SUBTOPICS = ["Optimizer: Weight decay L2", "Optimizer: decoupled weight decay (AdamW)"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

These two WD flavours look the same in the special case `g = 0` — both pull `theta` toward zero by the same multiplicative factor — but at a nonzero gradient they apply the decay at DIFFERENT points in the pipeline, and produce different updates.

**Atom A — weight-decay-l2-add.** `g <- g + lam * theta`, THEN `theta <- theta - lr * g`. Substituting: `theta_new = theta - lr * (g + lam * theta) = (1 - lr*lam) * theta - lr * g`. Here the WD shrinks theta and the gradient step are MIXED together — fine for vanilla SGD, interacts badly with Adam's adaptive denom.

**Atom B — weight-decay-decoupled.** `theta <- (1 - lr * lam) * theta`, THEN `theta <- theta - lr * g`. Substituting: `theta_new = (1 - lr*lam) * theta - lr * g`. For vanilla SGD (no adaptive scaling), this is ALGEBRAICALLY IDENTICAL to L2-add. The two atoms diverge ONLY when there's an adaptive denom in the middle (Adam → AdamW).

**The composite drill.** Implement BOTH update rules over a single SGD-style step. Show that with the SAME `(theta, g, lr, lam)`, L2-add and decoupled-WD produce the SAME `theta_new` (vanilla SGD case). Then change the step rule to include an adaptive denom — now they diverge.

### Composite Exercise — L2-add WD vs decoupled WD — different effective updates at nonzero g

**Atoms exercised together**: `weight-decay-l2-add`, `weight-decay-decoupled`

Implement two pure-function step rules, both returning a NEW tensor (no in-place):

1. `cx18_l2_add_step(theta, g, lr, lam)` — apply L2-add:
   - `g_eff = g + lam * theta`
   - return `theta - lr * g_eff`

2. `cx18_decoupled_step(theta, g, lr, lam)` — apply decoupled WD:
   - `theta_shrunk = (1 - lr * lam) * theta`
   - return `theta_shrunk - lr * g`

Then implement `cx18_decoupled_with_denom_step(theta, g, lr, lam, denom)` to show the interesting case: when the gradient is rescaled by a per-element `denom > 0` (mimicking Adam's `sqrt(v_hat) + eps`):
   - `theta_shrunk = (1 - lr * lam) * theta`
   - return `theta_shrunk - lr * (g / denom)`

And `cx18_l2_add_with_denom_step(theta, g, lr, lam, denom)`:
   - `g_eff = g + lam * theta`
   - return `theta - lr * (g_eff / denom)`   # ← the WD ALSO gets divided by denom.

The test shows: (a) without a denom (vanilla SGD), L2-add == decoupled identically; (b) WITH a denom, the two diverge — this is the AdamW vs Adam-L2 distinction.

In [ ]:
def cx18_l2_add_step(theta, g, lr, lam):
    # Atom A (weight-decay-l2-add): fold lam*theta INTO the grad.
    g_eff = g + lam * theta
    return theta - lr * g_eff

def cx18_decoupled_step(theta, g, lr, lam):
    # Atom B (weight-decay-decoupled): shrink theta directly, THEN apply the grad step.
    theta_shrunk = (1 - lr * lam) * theta
    return theta_shrunk - lr * g

def cx18_decoupled_with_denom_step(theta, g, lr, lam, denom):
    # Decay applies to theta unscaled; grad gets the adaptive rescale.
    theta_shrunk = (1 - lr * lam) * theta
    return theta_shrunk - lr * (g / denom)

def cx18_l2_add_with_denom_step(theta, g, lr, lam, denom):
    # Decay folded into the grad — so it ALSO gets divided by denom.
    g_eff = g + lam * theta
    return theta - lr * (g_eff / denom)


<details><summary>Show solution — cx18</summary>

```python
def cx18_l2_add_step(theta, g, lr, lam):
    # Atom A (weight-decay-l2-add): fold lam*theta INTO the grad.
    g_eff = g + lam * theta
    return theta - lr * g_eff

def cx18_decoupled_step(theta, g, lr, lam):
    # Atom B (weight-decay-decoupled): shrink theta directly, THEN apply the grad step.
    theta_shrunk = (1 - lr * lam) * theta
    return theta_shrunk - lr * g

def cx18_decoupled_with_denom_step(theta, g, lr, lam, denom):
    # Decay applies to theta unscaled; grad gets the adaptive rescale.
    theta_shrunk = (1 - lr * lam) * theta
    return theta_shrunk - lr * (g / denom)

def cx18_l2_add_with_denom_step(theta, g, lr, lam, denom):
    # Decay folded into the grad — so it ALSO gets divided by denom.
    g_eff = g + lam * theta
    return theta - lr * (g_eff / denom)
```

The key insight: with vanilla SGD (no adaptive denom), `theta - lr*(g + lam*theta) = (1 - lr*lam)*theta - lr*g` is just algebra — same expression two ways. With Adam's denom, the algebra breaks: in L2-add the `lam*theta` rides INSIDE the division (so the effective decay rate becomes lam/denom, parameter-dependent and curvature-rescaled), but in decoupled WD the `(1 - lr*lam)*theta` shrink happens OUTSIDE — a uniform, parameter-INdependent decay rate `lam`. That's why AdamW separates them.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx18'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx18',
        'subtopics': ["Optimizer: Weight decay L2", "Optimizer: decoupled weight decay (AdamW)"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()